# Validação das entidades

## Introdução

Após a conclusão da análise de requisitos, faz-se necessária a transição para a
modelagem conceitual de dados. Nesta etapa, a identificação das entidades
constitui o primeiro passo para a formalização estrutural do domínio, assegurando
que os principais elementos do problema de negócio sejam corretamente
representados no modelo.

Embora o documento de requisitos já apresente uma lista preliminar de entidades,
é imprescindível validá-las e refiná-las à luz do dataset e das regras de negócio
estabelecidas, evitando redundâncias, omissões ou inconsistências conceituais.

## Objetivo

Definir, de forma precisa e consistente, o conjunto final de entidades que
compõem o domínio do modelo de dados, garantindo aderência ao problema de
negócio e ao dataset utilizado.

In [2]:
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

data_directories = (Path("data/raw"), Path("../../data/raw"))
data_dir = next((path for path in data_directories if path.is_dir()), None)

if data_dir is None:
    raise FileNotFoundError(
        "Diretório data/raw não encontrado. Consulte data/README.md para obter os dados."
    )

customers = pd.read_csv(data_dir / "olist_customers_dataset.csv")
geolocation = pd.read_csv(data_dir / "olist_geolocation_dataset.csv")
order_items = pd.read_csv(data_dir / "olist_order_items_dataset.csv")
order_payments = pd.read_csv(data_dir / "olist_order_payments_dataset.csv")
order_reviews = pd.read_csv(data_dir / "olist_order_reviews_dataset.csv")
orders = pd.read_csv(data_dir / "olist_orders_dataset.csv")
products = pd.read_csv(data_dir / "olist_products_dataset.csv")
sellers = pd.read_csv(data_dir / "olist_sellers_dataset.csv")
product_category_name_translation = pd.read_csv(
    data_dir / "product_category_name_translation.csv"
)

## 1. Inventário das fontes de dados

O dataset é composto por nove arquivos. Oito representam entidades ou estruturas de dados incorporadas ao domínio analítico do projeto, incluindo a geolocalização de clientes e vendedores. O arquivo restante possui função auxiliar, sendo utilizado para a tradução das categorias de produtos.

In [3]:
tabelas = {
    "Clientes": customers,
    "Pedidos": orders,
    "Itens do Pedido": order_items,
    "Produtos": products,
    "Vendedores": sellers,
    "Pagamentos": order_payments,
    "Avaliações": order_reviews,
    "Geolocalização": geolocation,
    "Tradução de Categorias": product_category_name_translation,
}

entidades_do_dominio = {
    "Clientes",
    "Pedidos",
    "Itens do Pedido",
    "Produtos",
    "Vendedores",
    "Pagamentos",
    "Avaliações",
}

resumo_fontes = pd.DataFrame(
    [
        {
            "Fonte": nome,
            "Classificação": (
                "Entidade do domínio"
                if nome in entidades_do_dominio
                else "Tabela auxiliar"
            ),
            "Registros": len(df),
            "Atributos": df.shape[1],
            "Valores ausentes": int(df.isna().sum().sum()),
        }
        for nome, df in tabelas.items()
    ]
)

display(
    resumo_fontes.style.format(
        {
            "Registros": "{:,.0f}",
            "Atributos": "{:,.0f}",
            "Valores ausentes": "{:,.0f}",
        }
    ).hide(axis="index")
)

Fonte,Classificação,Registros,Atributos,Valores ausentes
Clientes,Entidade do domínio,"99,441",5,0
Pedidos,Entidade do domínio,"99,441",8,"4,908"
Itens do Pedido,Entidade do domínio,"112,650",7,0
Produtos,Entidade do domínio,"32,951",9,"2,448"
Vendedores,Entidade do domínio,"3,095",4,0
Pagamentos,Entidade do domínio,"103,886",5,0
Avaliações,Entidade do domínio,"99,224",7,"145,903"
Geolocalização,Tabela auxiliar,"1,000,163",5,0
Tradução de Categorias,Tabela auxiliar,71,2,0


## 2. Validação das entidades previstas

A validação compara as entidades definidas na análise de requisitos com as
fontes efetivamente disponíveis. Uma entidade é considerada validada quando há
uma fonte correspondente no dataset.

In [4]:
validacao_entidades = pd.DataFrame(
    {
        "Entidade prevista": sorted(entidades_do_dominio),
    }
)

validacao_entidades["Fonte encontrada"] = validacao_entidades[
    "Entidade prevista"
].isin(tabelas)
validacao_entidades["Resultado"] = validacao_entidades["Fonte encontrada"].map(
    {True: "Validada", False: "Não encontrada"}
)

display(
    validacao_entidades.drop(columns="Fonte encontrada")
    .style.hide(axis="index")
    .map(
        lambda valor: (
            "color: #137333; font-weight: bold"
            if valor == "Validada"
            else "color: #b3261e; font-weight: bold"
        ),
        subset=["Resultado"],
    )
)

quantidade_prevista = len(entidades_do_dominio)
quantidade_validada = int(validacao_entidades["Fonte encontrada"].sum())

display(
    Markdown(
        f"**Resultado:** {quantidade_validada} de {quantidade_prevista} "
        "entidades previstas foram encontradas no dataset."
    )
)

Entidade prevista,Resultado
Avaliações,Validada
Clientes,Validada
Itens do Pedido,Validada
Pagamentos,Validada
Pedidos,Validada
Produtos,Validada
Vendedores,Validada


**Resultado:** 7 de 7 entidades previstas foram encontradas no dataset.

## 3. Atributos identificados

A visão abaixo apresenta cada atributo em uma linha, facilitando a comparação
com o dicionário de dados e apoiando as próximas decisões sobre chaves,
relacionamentos e tipos de dados.

In [5]:
atributos_identificados = pd.DataFrame(
    [
        {
            "Fonte": nome,
            "Atributo": coluna,
            "Tipo inferido": str(df[coluna].dtype),
            "Valores ausentes": int(df[coluna].isna().sum()),
            "Valores únicos": int(df[coluna].nunique(dropna=True)),
        }
        for nome, df in tabelas.items()
        for coluna in df.columns
    ]
)

display(
    atributos_identificados.style.format(
        {
            "Valores ausentes": "{:,.0f}",
            "Valores únicos": "{:,.0f}",
        }
    ).hide(axis="index")
)

Fonte,Atributo,Tipo inferido,Valores ausentes,Valores únicos
Clientes,customer_id,str,0,"99,441"
Clientes,customer_unique_id,str,0,"96,096"
Clientes,customer_zip_code_prefix,int64,0,"14,994"
Clientes,customer_city,str,0,"4,119"
Clientes,customer_state,str,0,27
Pedidos,order_id,str,0,"99,441"
Pedidos,customer_id,str,0,"99,441"
Pedidos,order_status,str,0,8
Pedidos,order_purchase_timestamp,str,0,"98,875"
Pedidos,order_approved_at,str,160,"90,733"


## 4. Conclusão

As entidades e estruturas de dados previstas no documento de requisitos possuem representação no dataset. A Geolocalização foi incorporada ao domínio analítico por complementar as informações territoriais de clientes e vendedores e possibilitar análises geográficas. A Tradução de Categorias permanece classificada como tabela auxiliar, pois possui função de apoio à padronização e interpretação das categorias de produtos.

Com a identificação das entidades e estruturas de dados concluída, a próxima etapa consiste em validar seus atributos e definir quais campos atuarão como identificadores e chaves no modelo conceitual.